# Stage 2 Notebook 44 - Exp2OO Anchor + Hungarian 1-to-1 matching

**Targeted fix for matching instability.** Across NB39/40/41/42 the dynamic_k matcher labels K=4 priors as positive per GT lane in each batch -- but WHICH K=4 priors gets re-decided every batch based on the current IoU contest. The same prior is positive in batch A, negative in batch B. cls cannot learn discriminative scores from contradictory labels and collapses to near-uniform sigmoid (pos-neg gap < 0.01 across all four runs).

Exp2OO replaces dynamic_k with strict Hungarian 1-to-1: each GT lane is matched to exactly ONE prior, deterministically minimizing the cost matrix. The label per prior is now stable across batches (approximately -- still depends on which prior wins the cost contest, but with 1-to-1 there's a unique winner). DETR-style.

Single config diff vs Exp2KK (NB40):
- `lane_assigner: dynamic_k -> hungarian`

Plus the new `decoded_score_source = cls_x_mask` so we get the same three-way diagnostic as NB43 (`decoded_f1`, `decoded_cls_only_f1`, `decoded_mask_only_f1`) -- this lets us isolate matching-stability effects from the mask-consistency effects of NB43.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 20-epoch short run.
3. AMP keeps wall-clock ~ 30 minutes for 20 epochs at 3000 samples.
4. Output mirrored to notebook cell, Colab runtime log, Drive log file.
5. Do not rerun NB00. Independent of any prior NB; only depends on the dataset tar.

In [ ]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

In [ ]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp39_rmt_gca_anchor_hungarian_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

In [ ]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp39_rmt_gca_anchor_hungarian_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

## What to watch in Exp2OO training

Pass criteria at epoch 20:
- **`val/lane/decoded_cls_only_f1 >= 0.10`**. THIS is the hypothesis test for Exp2OO: with stable Hungarian labels, cls should now rank priors at least 2-3x better than NB40's 0.043 cls-only ranking. If it does, matching instability was the cls collapse cause.
- **`pos_score - neg_score >= 0.10`**. Direct measurement of cls separation.
- **`val/matched_line_iou >= 0.40`** (preserve geometry; Hungarian only matches num_GT priors per image so the rest get pure-negative gradient -- if this hurts geometry too much, dynamic_k was helping after all).
- **`val/lane/decoded_f1 >= 0.10`** with `cls_x_mask` ranking.

Failure signals:
- pos-neg gap still < 0.02: matching instability was not the root cause; the per-prior feature is fundamentally not discriminative. (Confirms that Exp2NN's mask-consistency angle is the right one.)
- matched_iou drops below 0.30: 1-to-1 matching is starving the geometry; only 5 priors per image get gradient on point regression. Switch to `match_cost_iou: 4.0` to bias matching toward best-fit priors.